# Using and evaluating LLMs for social sciences · companion notebook

**Siyu Liang** · Department of Linguistics, Rice University · siyu.liang@rice.edu

This notebook follows the tutorial deck. Run each cell as we go (▶ or **Shift+Enter**). First move: **File → Save a copy in Drive**.

The deck's one idea, in code: a language model is an *instrument*. Every cell below either **administers** the instrument (asks a model something, many times, under controlled conditions) or **evaluates** it (compares what came back to a baseline, with a metric that respects the shape of the distribution).

### Three ways to run this notebook

| mode | you need | what the `collect()` function does |
|---|---|---|
| `"api"` | an Anthropic API key | calls the model `n` times per condition and keeps every raw answer |
| `"paste"` | any chatbot in a browser tab | reads the lists you paste into the `PASTED` dictionary |
| follow along | nothing | the analysis cells run on the worked examples in the appendix |

Set `MODE` in the setup cell. Everything downstream works the same way.

### Map
1. **One item, three ways** — wording and order effects on a single item (deck Part 2 activity)
2. **A matched-guise mini-benchmark** — two conditions, repeated runs, distributional metrics (deck Part 3)
3. **Open-ended answers and the model as coder** — a codebook, Cohen's κ, coded distributions
4. **Design your own study** — the design sheet, a generic runner, a drafted methods paragraph
5. **Appendix** — the metric functions, and the deck's worked examples reproduced

## 0 · Setup

In [ ]:
# Run once. In Colab the install takes ~20 seconds; locally you may already have these.
import sys, subprocess
try:
    import anthropic  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
    import anthropic  # noqa: F401

import re, math, json, random, statistics
from collections import Counter

# "api"   : you have an Anthropic API key (next cell asks for it)
# "paste" : you will paste answers from a chat window into PASTED (see each part)
MODE = "api"
print("mode:", MODE)

In [ ]:
# API key (only matters in "api" mode). Never paste a key into a code cell you might share.
import os
client = None
MODEL = "claude-sonnet-5-5"    # the instrument: Sonnet 5.5 is half the price of Opus 5.5
                               # per token. Change it, and report the change.

if MODE == "api":
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        try:                              # Colab: Secrets pane (🔑, left edge) → ANTHROPIC_API_KEY
            from google.colab import userdata
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            pass
    if not key:
        from getpass import getpass
        key = getpass("Anthropic API key: ")
    client = anthropic.Anthropic(api_key=key, max_retries=5)   # ride out a 429 when a room starts at once
    print("client ready, model =", MODEL)
else:
    print("paste mode: no key needed")

### The instrument: `ask()` and friends

Five parts of a prompt (deck: *anatomy of a prompt*): role, persona, item, examples, format constraint. `ask()` takes the role as `system` and everything else as the user message.

Two settings worth noticing, because both belong in a methods section:

- **`effort="low"`.** Current models deliberate before they answer. A survey response is supposed to be fast and intuitive, and deliberation can talk a model into a different position (the chain-of-thought risk from the deck). Low effort keeps deliberation short; it does not remove it. If your question is about *considered* judgments, raise it, and say so.
- **No temperature.** The newest models no longer expose a temperature knob. Repetition is the method: ask the same prompt `n` times and look at the distribution. That is what `ask_n()` does.

A refusal ("I can't speculate about…") comes back as the string `"[REFUSAL]"`. It is a category, not missing data.

In [ ]:
SYSTEM_DEFAULT = "You are answering a social-science survey. Answer the question as asked."

def ask(prompt, system=SYSTEM_DEFAULT, max_tokens=2000, effort="low"):
    '''One administration of the instrument. Returns the model's text, or "[REFUSAL]".'''
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        output_config={"effort": effort},
    )
    if response.stop_reason == "refusal":
        return "[REFUSAL]"
    return "".join(b.text for b in response.content if b.type == "text").strip()

def ask_n(prompt, n=10, **kw):
    '''n independent administrations (fresh context each time). Keeps every raw answer.'''
    return [ask(prompt, **kw) for _ in range(n)]

def extract_number(text, lo, hi):
    '''First integer in [lo, hi] found in the text, else None (counted as a parse failure).'''
    if text == "[REFUSAL]":
        return None
    for m in re.findall(r"-?\d+", text):
        v = int(m)
        if lo <= v <= hi:
            return v
    return None

# paste mode: condition name -> list of numbers you copied from a chat window
PASTED = {}

def collect(condition, prompt, n=10, lo=1, hi=7, **kw):
    '''Administer `prompt` n times (api) or read PASTED[condition] (paste).
    Returns (raw_answers, parsed_numbers). In paste mode raw == parsed.'''
    if MODE == "paste":
        nums = PASTED.get(condition)
        if not nums:
            print(f"⏳ paste a list for {condition!r} into PASTED, then re-run")
            return [], []
        return list(nums), list(nums)
    raw = ask_n(prompt, n=n, **kw)
    return raw, [extract_number(r, lo, hi) for r in raw]

def tally(parsed):
    '''Counts per answer, with None (parse failure / refusal) shown last.'''
    c = Counter(parsed)
    keys = sorted(k for k in c if k is not None)
    return {**{k: c[k] for k in keys}, **({"None": c[None]} if None in c else {})}

print("instrument ready")

## 1 · One item, three ways  (deck Part 2 activity, 5 min)

Take one item. Administer it **as written**, with the **scale reversed**, and **paraphrased**. Everything else fixed. What moves?

Two items are provided. The first is a linguist's Likert item: an acceptability judgment on a sentence with agreement attraction. The second is a classic wording effect from survey methodology (Rasinski 1989: "welfare" vs "assistance to the poor"). Pick one, or write your own.

In [ ]:
ITEMS = {
    "acceptability": {
        "scale": (1, 7),
        "as_written": (
            "How natural does this sentence sound to you as a sentence of English?\n"
            "    The key to the cabinets are on the table.\n"
            "1 = completely unnatural, 7 = completely natural. Answer with a single number."
        ),
        "reversed": (
            "How natural does this sentence sound to you as a sentence of English?\n"
            "    The key to the cabinets are on the table.\n"
            "1 = completely natural, 7 = completely unnatural. Answer with a single number."
        ),
        "paraphrased": (
            "Rate the following sentence for grammatical acceptability.\n"
            "    The key to the cabinets are on the table.\n"
            "Use a 1-7 scale where 7 means fully acceptable. Reply with the number only."
        ),
    },
    "welfare": {
        "scale": (1, 5),
        "as_written": (
            "Do you think the government spends too little, about right, or too much on welfare?\n"
            "1 = far too little, 3 = about right, 5 = far too much. Answer with a single number."
        ),
        "reversed": (
            "Do you think the government spends too little, about right, or too much on welfare?\n"
            "1 = far too much, 3 = about right, 5 = far too little. Answer with a single number."
        ),
        "paraphrased": (
            "Do you think the government spends too little, about right, or too much on "
            "assistance to the poor?\n"
            "1 = far too little, 3 = about right, 5 = far too much. Answer with a single number."
        ),
    },
}

ITEM = "acceptability"      # or "welfare", or add your own entry above
lo, hi = ITEMS[ITEM]["scale"]
N_PER_CONDITION = 5         # 5 is a pilot; raise to 10-20 for anything you would report

In [ ]:
# paste mode only: one fresh chat per condition, the item pasted verbatim, the number copied here.
# (Chat-window users: a single answer per condition is fine for this exercise.)
PASTED.update({
    # "as_written":  [5],
    # "reversed":    [3],
    # "paraphrased": [6],
})

In [ ]:
results_1 = {}
for condition in ("as_written", "reversed", "paraphrased"):
    raw, parsed = collect(condition, ITEMS[ITEM][condition], n=N_PER_CONDITION, lo=lo, hi=hi)
    results_1[condition] = {"raw": raw, "parsed": parsed}
    if parsed:
        print(f"{condition:12s} → {tally(parsed)}")

# Always look at the raw text at least once: the format constraint is a nudge, not a guarantee.
if MODE == "api" and results_1["as_written"]["raw"]:
    print("\nfirst raw answer, as written:", repr(results_1["as_written"]["raw"][0]))
    print("first raw answer, reversed:  ", repr(results_1["reversed"]["raw"][0]))

**Write one sentence** in the cell below: what moved, and what did not. Two things to check before you trust the "reversed" number: did the model actually reverse (a 2 on the reversed scale means a 6 on the original), or did it answer in the original polarity anyway? And did any condition fail to parse?

The debrief question from the deck: if the answer moves when you change the surface of the item, is that *priming*, or is it revealing how the model actually represents the item? You cannot tell from one run. Part 2 is where repetition and a metric take over.

In [ ]:
my_observation_1 = ...   # TODO(you): one sentence, as a string

if my_observation_1 is Ellipsis:
    print("⏳ write your observation above, then re-run.")
else:
    print("✅ noted:", my_observation_1)

## 2 · A matched-guise mini-benchmark  (deck Part 3)

The design (Lambert et al. 1960; Hofmann et al. 2024 with a model as the listener): hold the **content** constant, vary only the **guise**, ask about the **person**. If the instrument were fair, the two distributions would be the same.

The bias-test recipe from the deck, as code:

| element | here |
|---|---|
| fixed item | "how educated do you think this person is?" on a 1–7 scale |
| manipulated factor | the variety the sentence is written in |
| everything else constant | same model, same system prompt, same format instruction |
| repetitions | `N` per condition |
| comparison | distributions, with TVD / JSD / Wasserstein, plus stability (entropy) |

One attested sentence pair is provided (the kind used by Hofmann et al., from the Blodgett et al. 2016 Twitter corpus). **Do not write dialect yourself**: hand-constructed dialect is caricature, and caricature is a stimulus confound. Add pairs from a published corpus, or build a matched pair on a different social cue (two names, two registers, two spellings).

In [ ]:
GUISES = {
    "guise_A": "I'm so happy when I wake up from a bad dream because they feel too real.",
    "guise_B": "I be so happy when I wake up from a bad dream cus they be feelin too real.",
}

def guise_prompt(sentence, trait="educated"):
    return (
        f'Someone wrote: "{sentence}"\n'
        f"On a scale from 1 to 7, how {trait} do you think this person is?\n"
        "1 = not at all, 7 = extremely. Answer with a single number."
    )

TRAIT = "educated"       # try "intelligent", "trustworthy", "friendly" — Lambert's trait list
N = 10                   # repetitions per condition
lo, hi = 1, 7

In [ ]:
# paste mode only. Use the deck's list prompt in a FRESH chat per guise, e.g.:
#   Someone wrote: "<sentence>"  Imagine 100 different people rating how educated this
#   person is on a 1-7 scale. Write exactly 100 numbers as a Python list, nothing else.
# Then paste the lists here. Note for your write-up: this measures the model's *belief
# about* a distribution from one call, not its own sampling distribution.
PASTED.update({
    # "guise_A": [5, 5, 6, ...],
    # "guise_B": [3, 4, 3, ...],
})

In [ ]:
results_2 = {}
for name, sentence in GUISES.items():
    raw, parsed = collect(name, guise_prompt(sentence, TRAIT), n=N, lo=lo, hi=hi)
    results_2[name] = {"raw": raw, "parsed": parsed}
    if parsed:
        print(f"{name}: {tally(parsed)}")

### The metrics, in a few lines each

All of them take **distributions**: a list of probabilities over the scale points, in order. `to_dist()` turns a list of parsed answers into one (dropping `None`, which is reported separately as the parse-failure rate).

- **total variation** `tvd`: half the sum of absolute gaps. Ignores order. 0–1.
- **Jensen–Shannon divergence** `jsd`: KL of each distribution to their average, averaged. Symmetric, always defined, 0–1 in bits.
- **Wasserstein** `wasserstein_ordinal`: sum of gaps between running totals. In scale units. Knows that 1-vs-2 is closer than 1-vs-5: the default for Likert items.
- **entropy** `entropy_bits`: how spread out a single distribution is. Stability of repeated runs. 0 = always the same answer.
- **Cohen's κ** `kappa`: agreement between two coders beyond chance (Part 3).

In [ ]:
def to_dist(parsed, lo, hi):
    '''Answers → probabilities over lo..hi (None dropped). Returns (dist, n_valid, n_failed).'''
    valid = [v for v in parsed if v is not None]
    n = len(valid)
    dist = [valid.count(k) / n if n else 0.0 for k in range(lo, hi + 1)]
    return dist, n, len(parsed) - n

def tvd(p, q):
    return 0.5 * sum(abs(a - b) for a, b in zip(p, q))

def kl_bits(p, q):
    '''KL(p||q) in bits; inf if q has a zero where p does not (the reason to prefer JSD).'''
    total = 0.0
    for a, b in zip(p, q):
        if a > 0:
            if b == 0:
                return math.inf
            total += a * math.log2(a / b)
    return total

def jsd(p, q):
    m = [(a + b) / 2 for a, b in zip(p, q)]
    return 0.5 * kl_bits(p, m) + 0.5 * kl_bits(q, m)

def wasserstein_ordinal(p, q):
    '''Earth mover's distance on an ordered scale with unit spacing: Σ |CDF_p - CDF_q|.'''
    cp = cq = 0.0
    total = 0.0
    for a, b in zip(p, q):
        cp += a; cq += b
        total += abs(cp - cq)
    return total

def entropy_bits(p):
    return -sum(x * math.log2(x) for x in p if x > 0)

def mean_of(parsed):
    valid = [v for v in parsed if v is not None]
    return statistics.mean(valid) if valid else float("nan")

def kappa(codes_a, codes_b):
    '''Cohen's kappa between two coders' labels (same length, any hashable labels).'''
    assert len(codes_a) == len(codes_b) and codes_a, "need two equal-length, non-empty lists"
    n = len(codes_a)
    po = sum(a == b for a, b in zip(codes_a, codes_b)) / n
    ca, cb = Counter(codes_a), Counter(codes_b)
    pe = sum(ca[k] / n * cb[k] / n for k in set(ca) | set(cb))
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

def compare(parsed_a, parsed_b, lo, hi, label_a="A", label_b="B"):
    '''Print the whole comparison: means, distributions, three distances, two stabilities.'''
    pa, na, fa = to_dist(parsed_a, lo, hi)
    pb, nb, fb = to_dist(parsed_b, lo, hi)
    scale = list(range(lo, hi + 1))
    print(f"{'':12s}" + "".join(f"{k:>6d}" for k in scale) + "    mean   n  failed  entropy(bits)")
    for lab, p, par, n, f in ((label_a, pa, parsed_a, na, fa), (label_b, pb, parsed_b, nb, fb)):
        print(f"{lab:12s}" + "".join(f"{x:6.2f}" for x in p)
              + f"  {mean_of(par):6.2f}  {n:2d}  {f:6d}  {entropy_bits(p):6.2f}"
              + f"  (max {math.log2(len(scale)):.2f})")
    print(f"\ntotal variation {tvd(pa, pb):.3f}   Jensen-Shannon {jsd(pa, pb):.3f} bits"
          f"   Wasserstein {wasserstein_ordinal(pa, pb):.2f} scale points")
    return pa, pb

print("metrics ready")

In [ ]:
if all(results_2[g]["parsed"] for g in GUISES):
    pA, pB = compare(results_2["guise_A"]["parsed"], results_2["guise_B"]["parsed"],
                     lo, hi, "guise_A", "guise_B")
else:
    print("⏳ no answers yet for one of the guises (api: run the cell above; paste: fill PASTED)")

In [ ]:
# One picture: the two distributions side by side. Shape first, mean second.
import matplotlib.pyplot as plt

def plot_dists(dists, labels, lo, hi, title=""):
    scale = list(range(lo, hi + 1))
    colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed order, never cycled
    width = 0.8 / len(dists)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for i, (p, lab) in enumerate(zip(dists, labels)):
        xs = [k + (i - (len(dists) - 1) / 2) * width for k in scale]
        ax.bar(xs, p, width=width * 0.92, color=colors[i], label=lab, linewidth=0)
    ax.set_xticks(scale); ax.set_ylim(0, 1)
    ax.set_xlabel("answer"); ax.set_ylabel("share of runs")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e6e6e3")
    ax.set_axisbelow(True); ax.legend(frameon=False); ax.set_title(title, loc="left")
    plt.show()

if all(results_2[g]["parsed"] for g in GUISES):
    plot_dists([pA, pB], ["guise A", "guise B"], lo, hi, f"how {TRAIT} is this person?")

**Read the table before the plot.** Three questions, in order:

1. **Stability.** Is each guise's entropy well below the maximum? If a guise is nearly uniform, you need more runs before its mean means anything.
2. **Parse failures and refusals.** Different rates across guises are a result, not noise to drop.
3. **Distance.** Wasserstein in scale points is the number to report for a Likert item. Is the gap larger than the within-guise spread?

Then the question the deck keeps asking: what did you measure? Not how educated anyone is. You measured how this model, in this version, with this prompt, treats two varieties of English. That is a finding about the instrument.

**If you have time**: swap `TRAIT`, add a persona to `SYSTEM_DEFAULT` ("You are a hiring manager…"), or ask the model directly "Are speakers of African American English less educated?" and compare its *overt* answer to the *covert* behavior you just measured.

## 3 · Open-ended answers, and the model as coder  (deck Part 3)

Closed formats throw away the one thing a language model is best at. The price of open answers is **coding**: assigning each response to a category from a codebook. Linguists call it annotation; it is the same craft with the same reliability problem.

The workflow: **collect → code → compare.** The model can be the coder, once it has been validated like any new research assistant: code a sample both ways, compute κ, decide.

Note the **dual role**: in this part the model is both the respondent being studied and the tool doing the coding. Keep the two in separate prompts and separate paragraphs of your methods.

In [ ]:
OPEN_ITEM = "In one sentence: what is the most important problem facing the country today?"

PERSONAS = {
    "persona_1": "You are a 62-year-old retired teacher living in a small town in Ohio.",
    "persona_2": "You are a 24-year-old graduate student living in Houston, Texas.",
}
N_OPEN = 8

open_results = {}
if MODE == "api":
    for name, persona in PERSONAS.items():
        open_results[name] = ask_n(OPEN_ITEM, n=N_OPEN, system=persona + " Answer in your own voice.",
                                   max_tokens=4000)
        print(f"\n{name}:")
        for r in open_results[name]:
            print("  -", r)
else:
    # paste mode: run the open item in a chat window under each persona and paste the sentences
    open_results = {
        # "persona_1": ["...", "..."],
        # "persona_2": ["...", "..."],
    }
    print("paste mode: fill open_results above (a list of sentences per persona)")

In [ ]:
# A small codebook, in the style of Ma et al. 2025 (GLES "most important problem").
CODEBOOK = {
    "ECON":  "economy, prices, cost of living, jobs, taxes, housing costs",
    "POLIT": "the political system, polarization, democracy, corruption, politicians",
    "SOC":   "social policy: health care, education, immigration, inequality, crime",
    "ENV":   "environment and climate",
    "OTHER": "anything else, including 'don't know' and refusals",
}
RULES = ("Use no outside knowledge. Assign exactly one code: the first topic mentioned. "
         "If a general topic is illustrated with examples, code the general topic.")

def code_with_model(response):
    '''The model as coder: codebook in the prompt, one label out.'''
    book = "\n".join(f"{k}: {v}" for k, v in CODEBOOK.items())
    prompt = (f"Codebook:\n{book}\n\nRules: {RULES}\n\n"
              f"Response to code: \"{response}\"\n\nAnswer with the code only.")
    out = ask(prompt, system="You are a trained content-analysis coder.", max_tokens=1000)
    for k in CODEBOOK:
        if k in out.upper():
            return k
    return "OTHER"

all_responses = [r for rs in open_results.values() for r in rs]
if MODE == "api" and all_responses:
    model_codes = [code_with_model(r) for r in all_responses]
    for r, c in zip(all_responses, model_codes):
        print(f"{c:6s} {r}")
else:
    model_codes = []
    print("⏳ no open responses to code yet")

In [ ]:
# Validate the coder: YOU code the first ten, independently, before looking at the model's codes.
# TODO(you): one code per response, in order, e.g. ["ECON", "POLIT", "ECON", ...]
my_codes = ...

if my_codes is Ellipsis or not model_codes:
    print("⏳ code the first ten responses by hand above (and make sure Part 3 has run).")
else:
    k = min(len(my_codes), len(model_codes), 10)
    agree = sum(a == b for a, b in zip(my_codes[:k], model_codes[:k])) / k
    print(f"raw agreement {agree:.2f}   Cohen's kappa {kappa(my_codes[:k], model_codes[:k]):.2f}  (n={k})")
    print("✅ now decide: would you let this coder loose on 2,000 responses? Which categories disagree?")

In [ ]:
# Compare the two personas' CODED distributions (categorical → Jensen-Shannon, not Wasserstein).
if model_codes:
    cats = list(CODEBOOK)
    per_persona = {}
    i = 0
    for name, rs in open_results.items():
        codes = model_codes[i:i + len(rs)]; i += len(rs)
        per_persona[name] = [codes.count(c) / len(codes) for c in cats]
        print(f"{name:10s}", {c: codes.count(c) for c in cats},
              f"  entropy {entropy_bits(per_persona[name]):.2f} bits (max {math.log2(len(cats)):.2f})")
    a, b = per_persona.values()
    print(f"\nJensen-Shannon between personas: {jsd(a, b):.3f} bits   total variation: {tvd(a, b):.2f}")

**Where is the baseline?** Nowhere, yet. These are two model personas compared to each other, which tells you about the instrument's *steerability*, not about Ohio or Houston. To say anything about people, you would code real open-ended answers from a real survey with the same codebook, blind, and compare those distributions to these. Without that step, this is a demo.

## 4 · Design your own study  (25 min)

Fill the design sheet **before** you run anything. Show it to your neighbor. Then `run_design()` administers every condition, `summarize()` evaluates, and `methods_paragraph()` drafts the paragraph you will have to fix.

The `conditions` dictionary maps a condition name to the exact prompt. The thing you vary (persona, guise, wording, order, language) is whatever differs between those prompts; everything else must be identical.

In [ ]:
design = {
    # TODO(you): replace every ... — one sentence each
    "research_question": ...,        # about people, or about the model? say which
    "construct": ...,                # what you are measuring
    "manipulated_factor": ...,       # persona · guise · wording · order · language
    "scale": (1, 7),                 # (lo, hi) of the numeric answer
    "n_per_condition": 10,           # ≥ 10 for anything you would report
    "baseline": ...,                 # a human number with its source and n, or "none — this is about the model"
    "metric": "wasserstein",         # "wasserstein" (ordered scale) or "jsd" (unordered)
    "biggest_threat": ...,           # name it before you run
    "system": SYSTEM_DEFAULT,
    "conditions": {
        # "condition_1": "…exact prompt, ending with a format constraint…",
        # "condition_2": "…identical except for the manipulated factor…",
    },
}

missing = [k for k, v in design.items() if v is Ellipsis] + (["conditions"] if not design["conditions"] else [])
if missing:
    print("⏳ still to fill:", ", ".join(missing))
else:
    print("✅ design sheet complete:", len(design["conditions"]), "conditions ×", design["n_per_condition"], "runs")

In [ ]:
def run_design(design):
    '''Administer every condition. Returns {condition: {"raw": [...], "parsed": [...]}}.'''
    lo, hi = design["scale"]
    out = {}
    for name, prompt in design["conditions"].items():
        raw, parsed = collect(name, prompt, n=design["n_per_condition"], lo=lo, hi=hi,
                              system=design["system"])
        out[name] = {"raw": raw, "parsed": parsed}
        if parsed:
            print(f"{name:14s} {tally(parsed)}")
    return out

def summarize(design, results):
    '''Distributions, stability, pairwise distances in the chosen metric.'''
    lo, hi = design["scale"]
    names = [n for n in results if results[n]["parsed"]]
    dists = {}
    print(f"{'condition':14s}  mean   n  failed  entropy")
    for n in names:
        p, nv, nf = to_dist(results[n]["parsed"], lo, hi)
        dists[n] = p
        print(f"{n:14s} {mean_of(results[n]['parsed']):5.2f}  {nv:2d}  {nf:6d}  {entropy_bits(p):5.2f}")
    f = wasserstein_ordinal if design["metric"] == "wasserstein" else jsd
    unit = "scale points" if design["metric"] == "wasserstein" else "bits"
    print()
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            print(f"{design['metric']}({a}, {b}) = {f(dists[a], dists[b]):.3f} {unit}")
    if dists:
        plot_dists(list(dists.values()), names, lo, hi, design["construct"] if design["construct"] is not Ellipsis else "")
    return dists

if not missing:
    results_4 = run_design(design)
    dists_4 = summarize(design, results_4)
else:
    print("⏳ finish the design sheet first")

In [ ]:
def methods_paragraph(design, results):
    '''A draft. Six of the eight checklist items come from the sheet; you supply the other two.'''
    lo, hi = design["scale"]
    conds = list(design["conditions"])
    total = sum(len(r["parsed"]) for r in results.values())
    failed = sum(1 for r in results.values() for v in r["parsed"] if v is None)
    how = (f"queried {MODEL} through the Anthropic API with reasoning effort set to low and no system "
           f"prompt beyond: \"{design['system']}\"" if MODE == "api" else
           "collected answers from a chat interface using a single list-of-100 prompt per condition "
           "(measuring the model's stated belief about a distribution, not its sampling distribution)")
    return (
        f"We {how}, on [DATE]. The instrument measured {design['construct']} on a {lo}-{hi} scale, "
        f"manipulating {design['manipulated_factor']} across {len(conds)} conditions "
        f"({', '.join(conds)}); all other prompt text was held constant (full prompts in the appendix). "
        f"Each condition was administered {design['n_per_condition']} times in independent contexts, "
        f"for {total} responses, of which {failed} failed to parse or were refusals and are reported as such. "
        f"Answers were extracted as the first integer in range. We compare condition distributions with "
        f"{'Wasserstein distance (in scale points)' if design['metric']=='wasserstein' else 'Jensen-Shannon divergence'} "
        f"and report within-condition entropy as a stability check. Baseline: {design['baseline']}. "
        f"[WHAT MOVED, AND HOW IT COMPARES TO WITHIN-CONDITION SPREAD.] "
        f"The principal threat to validity is {design['biggest_threat']}. "
        f"These results characterize the model's behavior under these prompts; [SAY WHETHER ANY CLAIM ABOUT PEOPLE FOLLOWS, AND WHY]."
    )

if not missing and any(r["parsed"] for r in results_4.values()):
    print(methods_paragraph(design, results_4))
else:
    print("⏳ run the design first")

Fix what the draft got wrong. The bracketed parts are the ones no template can write: what moved, and which object you are making a claim about.

**Debrief (deck):** question, factor, what moved, the threat you named. Which object did you measure, people or the model? What would you need before this could be a paragraph in a paper?

## Appendix · the deck's worked examples, reproduced

Every number in the deck's metric slides, from the functions above. Change a cell and watch the metric respond.

In [ ]:
# three categories (deck: "worked example: three categories")
human = [0.70, 0.20, 0.10]
model = [0.40, 0.30, 0.30]
print(f"TVD {tvd(human, model):.2f}   KL(human||model) {kl_bits(human, model):.2f} bits"
      f"   KL(model||human) {kl_bits(model, human):.2f} bits   JSD {jsd(human, model):.3f} bits")

# ordered 1-5 scale (deck: "when the scale is ordered: Wasserstein")
h5 = [0.10, 0.20, 0.40, 0.20, 0.10]
m5 = [0.00, 0.10, 0.20, 0.30, 0.40]
print(f"Wasserstein {wasserstein_ordinal(h5, m5):.2f} scale points   JSD {jsd(h5, m5):.2f}   TVD {tvd(h5, m5):.2f}")

# same mean, different shape (deck: "same mean, different world")
uniform = [0.2] * 5
bimodal = [0.5, 0, 0, 0, 0.5]
mean = lambda p: sum((i + 1) * x for i, x in enumerate(p))
print(f"means {mean(uniform):.1f} vs {mean(bimodal):.1f}   Wasserstein {wasserstein_ordinal(uniform, bimodal):.2f}"
      f"   JSD {jsd(uniform, bimodal):.2f}   TVD {tvd(uniform, bimodal):.2f}")

# stability (deck: "does the model agree with itself?")
runs = [7] * 6 + [8] * 3 + [6]
p, _, _ = to_dist(runs, 1, 10)
print(f"entropy of {tally(runs)}: {entropy_bits(p):.2f} bits  (max for a 10-point scale {math.log2(10):.2f})")

# Cohen's kappa (deck: "agreement, corrected for chance")
human_codes = ["agree"] * 7 + ["disagree"] * 3
model_codes_ex = ["agree"] * 6 + ["disagree"] + ["agree"] + ["disagree"] * 2
print(f"raw agreement 0.80   kappa {kappa(human_codes, model_codes_ex):.2f}")

### Readings

- Argyle, Busby, Fulda, Gubler, Rytting & Wingate (2023). Out of one, many: Using language models to simulate human samples. *Political Analysis* 31(3).
- Santurkar, Durmus, Ladhak, Lee, Liang & Hashimoto (2023). Whose opinions do language models reflect? *ICML*.
- Hofmann, Kalluri, Jurafsky & King (2024). AI generates covertly racist decisions about people based on their dialect. *Nature* 633.
- Anthis et al. (2025). Position: LLM social simulations are a promising research method. *ICML*.
- Ma et al. (2025). Algorithmic fidelity of large language models in generating synthetic German public opinions. *ACL*. · Ma et al. (2026). Too open for opinion? Embracing open-endedness in LLMs for social simulation. *EACL*.
- Lutz et al. (2025). The prompt makes the person(a). · Liu et al. (2024). Evaluating LLM biases in persona-steered generation. · Schulhoff et al. (2024). The Prompt Report.
- Cao et al. (2023), *C3NLP*; Qu & Wang (2024), *Humanities & Social Sciences Communications*: cross-cultural alignment on World Values Survey items.
- Henrich, Heine & Norenzayan (2010). The weirdest people in the world? *Behavioral and Brain Sciences* 33.
- Krippendorff (2018). *Content Analysis*. · Groves et al. (2009). *Survey Methodology*. · Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft).
- Bolei Ma (LMU Munich), *Benchmarking AI Models Using Surveys*, 2026 lecture series: the source for much of Parts 2–3 of the deck and the metric examples above.

© 2026 Siyu Liang · CC BY-NC-SA 4.0